In [5]:
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.metrics import make_scorer, accuracy_score
import numpy as np
import xgboost as xgb
import json
import pandas as pd
from pandas import json_normalize
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from datetime import datetime
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from textblob import TextBlob
import re
from sklearn.ensemble import RandomForestClassifier
from gensim.models import Word2Vec
from nltk.corpus import stopwords


## Best pipeline (XGBoost)

In [ ]:
# ===============================
# JSONL LOADING
# ===============================

def load_jsonl_skip_bad(path):
    data_list = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                data_list.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return json_normalize(data_list)

train_data = pd.read_json('train.jsonl', lines=True)
train_data = json_normalize(train_data.to_dict(orient='records'))

kaggle_data = pd.read_json('kaggle_test.jsonl', lines=True)
kaggle_data = json_normalize(kaggle_data.to_dict(orient='records'))

X_train = train_data.drop('label', axis=1)
y_train = train_data['label']

X_kaggle = kaggle_data

print("Chargement OK")

Chargement OK


In [7]:
def create_advanced_features(df_input):
    df = df_input.copy()
    
    default_int_series = pd.Series(0, index=df.index)
    default_bool_series = pd.Series(False, index=df.index)
    
    df['user.followers_count'] = df.get('user.followers_count', default_int_series).fillna(0)
    df['user.friends_count'] = df.get('user.friends_count', default_int_series).fillna(0)
    df['user.listed_count'] = df.get('user.listed_count', default_int_series).fillna(0)
    df['user.favourites_count'] = df.get('user.favourites_count', default_int_series).fillna(0)
    df['user.statuses_count'] = df.get('user.statuses_count', default_int_series).fillna(0)
    df['retweet_count'] = df.get('retweet_count', default_int_series).fillna(0)
    df['favorite_count'] = df.get('favorite_count', default_int_series).fillna(0)
    df['quote_count'] = df.get('quote_count', default_int_series).fillna(0) # New : Quote Count
    df['reply_count'] = df.get('reply_count', default_int_series).fillna(0) # New : Reply Count
    
    # Get the dates
    df['user_created_at_dt'] = pd.to_datetime(df.get('user.created_at'), errors='coerce')
    ref_date = pd.to_datetime('now', utc=True)
    df['account_age_days'] = (ref_date - df['user_created_at_dt']).dt.days
    df['account_age_days'] = df['account_age_days'].fillna(0)
    
    # Extraction temporal indic
    df['created_at_dt'] = pd.to_datetime(df.get('created_at'), errors='coerce')
    df['tweet_hour'] = df['created_at_dt'].dt.hour.fillna(-1)
    df['tweet_is_weekend'] = df['created_at_dt'].dt.dayofweek.isin([5, 6]).fillna(False).astype(int)

    # Profile
    df['is_default_profile'] = df.get('user.default_profile', default_bool_series).fillna(False).astype(int)
    df['is_default_image'] = df.get('user.default_profile_image', default_bool_series).fillna(False).astype(int)
    df['is_verified'] = df.get('user.verified', default_bool_series).fillna(False).astype(int)
    df['is_protected'] = df.get('user.protected', default_bool_series).fillna(False).astype(int)
    df['has_url'] = df.get('user.url', pd.Series(False, index=df.index)).notna().astype(int)

    # Content
    def count_entities(x):
        if isinstance(x, list) or (isinstance(x, pd.Series) and x.dtype == object): return len(x)
        return 0

    df['num_urls'] = df.get('entities.urls', default_int_series).apply(count_entities)
    df['num_hashtags'] = df.get('entities.hashtags', default_int_series).apply(count_entities)
    df['num_mentions'] = df.get('entities.user_mentions', default_int_series).apply(count_entities)
    df['has_media'] = df.get('extended_entities.media', default_bool_series).notna().astype(int)

    # RATIOS
    followers = df['user.followers_count']
    friends = df['user.friends_count']
    listed = df['user.listed_count']
    statuses = df['user.statuses_count']
    df['ratio_followers_friends'] = followers / (friends + 1)
    df['ratio_listed_followers'] = listed / (followers + 1)
    df['reciprocity_score'] = (friends - followers) / (friends + followers + 1)
    df['tweets_per_day'] = statuses / (df['account_age_days'] + 1)
    df['ratio_mention_status'] = df['num_mentions'] / (statuses + 1)
    total_engagement = df['retweet_count'] + df['favorite_count'] + df['quote_count'] + df['reply_count']
    df['total_tweet_engagement'] = total_engagement / (followers + 1)

    # Lenght
    df['final_text'] = df.get('extended_tweet.full_text', df.get('text', pd.Series(''))).fillna('')
    df['final_text'] = df['final_text'].where(df['final_text'] != '', df.get('text', '')).fillna('')
    df['text_length'] = df['final_text'].astype(str).apply(len)
    df['bio_length'] = df.get('user.description', '').astype(str).apply(len)

    features_to_keep = [
        'user.followers_count', 'user.friends_count', 'user.listed_count', 
        'user.favourites_count', 'user.statuses_count',
        'retweet_count', 'favorite_count', 'quote_count', 'reply_count',
        'ratio_followers_friends', 'ratio_listed_followers', 'tweets_per_day', 'account_age_days',
        'reciprocity_score', 'ratio_mention_status', 'total_tweet_engagement',
        'is_verified', 'is_default_profile', 'is_default_image', 'is_geo_enabled',
        'is_protected', 'has_url',
        'tweet_hour', 'tweet_is_weekend',
        'text_length', 'bio_length', 
        'num_urls', 'num_hashtags', 'num_mentions', 'has_media',
    ]
    
    final_cols = [c for c in features_to_keep if c in df.columns]
    
    return df[final_cols].fillna(0)

In [8]:
def create_nlp_features(df_train, df_test, y_train):
    # ----------------------------------------
    # Prep of the texts
    # ----------------------------------------

    french_stopwords = stopwords.words("french")

    train_bio = df_train.get('user.description', pd.Series([''] * len(df_train))).fillna('').astype(str)
    test_bio = df_test.get('user.description', pd.Series([''] * len(df_test))).fillna('').astype(str)

    def get_final_text(df):
        text = df.get('text', pd.Series([''] * len(df))).fillna('')
        full_text = df.get('extended_tweet.full_text', text).fillna(text)
        return full_text.astype(str)
        
    train_text = get_final_text(df_train)
    test_text = get_final_text(df_test)

    def clean_text(text):
        text = text.lower()
        text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
        return text

    # ----------------------------------------
    # TF-IDF + log_reg on tweet
    # ----------------------------------------
    train_text_clean = train_text.apply(clean_text)
    test_text_clean = test_text.apply(clean_text)
    tfidf = TfidfVectorizer(max_features=1000, stop_words=french_stopwords, ngram_range=(2, 5), analyzer='char_wb', lowercase=False)
    X_train_tfidf_tweet = tfidf.fit_transform(train_text_clean)
    X_test_tfidf_tweet = tfidf.transform(test_text_clean)
    log_reg = LogisticRegression(solver='sag', random_state=42)
    log_reg.fit(X_train_tfidf_tweet, y_train.astype(int))
    train_tweet_proba = log_reg.predict_proba(X_train_tfidf_tweet)[:, 1]
    test_tweet_proba = log_reg.predict_proba(X_test_tfidf_tweet)[:, 1]    

    # ----------------------------------------
    # TF-IDF + log_reg on bio
    # ----------------------------------------
    train_bio_clean = train_bio.apply(clean_text)
    test_bio_clean = test_bio.apply(clean_text)
    tfidf = TfidfVectorizer(max_features=1000, stop_words=french_stopwords, ngram_range=(2, 5), analyzer='char_wb', lowercase=False)
    X_train_tfidf = tfidf.fit_transform(train_bio_clean)
    X_test_tfidf = tfidf.transform(test_bio_clean)
    log_reg = LogisticRegression(solver='liblinear', random_state=42)
    log_reg.fit(X_train_tfidf, y_train.astype(int))
    train_bio_proba = log_reg.predict_proba(X_train_tfidf)[:, 1]
    test_bio_proba = log_reg.predict_proba(X_test_tfidf)[:, 1]

    # ----------------------------------------
    # Sentiment analysis
    # ----------------------------------------
    def get_sentiment(text):
        try:
            analysis = TextBlob(text)
            return pd.Series({'polarity': analysis.sentiment.polarity, 'subjectivity': analysis.sentiment.subjectivity})
        except:
            return pd.Series({'polarity': 0.0, 'subjectivity': 0.0})
    train_sentiment = train_text.apply(get_sentiment)
    test_sentiment = test_text.apply(get_sentiment)

    # ----------------------------------------
    # Fusion of the features
    # ----------------------------------------
    df_train_nlp = pd.DataFrame({
        'meta_bio_proba': train_bio_proba,
        'tweet_polarity': train_sentiment['polarity'],
        'tweet_subjectivity': train_sentiment['subjectivity'],
        'meta_tweet_proba': train_tweet_proba
    })
    df_test_nlp = pd.DataFrame({
        'meta_bio_proba': test_bio_proba,
        'tweet_polarity': test_sentiment['polarity'],
        'tweet_subjectivity': test_sentiment['subjectivity'],
        'meta_tweet_proba': test_tweet_proba
    })

    return df_train_nlp, df_test_nlp

In [ ]:
X_train_advanced = create_advanced_features(X_train)
X_kaggle_advanced = create_advanced_features(X_kaggle)

y_train_clean = y_train.astype(int)
X_train_nlp, X_kaggle_nlp = create_nlp_features(X_train, X_kaggle, y_train_clean)
X_train_advanced = pd.concat([X_train_advanced, X_train_nlp], axis=1)
X_kaggle_advanced = pd.concat([X_kaggle_advanced, X_kaggle_nlp], axis=1)

print(f"\nFeatures combined ({len(X_train_advanced.columns)}):")
print(list(X_train_advanced.columns))

C:\Users\hp1ma\AppData\Local\Temp\ipykernel_14780\2501392516.py:18: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['user_created_at_dt'] = pd.to_datetime(df.get('user.created_at'), errors='coerce')
C:\Users\hp1ma\AppData\Local\Temp\ipykernel_14780\2501392516.py:18: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['user_created_at_dt'] = pd.to_datetime(df.get('user.created_at'), errors='coerce')
c:\Users\hp1ma\anaconda3\envs\inf554\Lib\site-packages\sklearn\feature_extraction\text.py:543: UserWarning: The parameter 'stop_words' will not be used since 'analyzer' != 'word'
  warnings.warn(
c:\Users\hp1ma\anaconda3\envs\inf554\Lib\site-packages\sklearn\feature_extraction\text.py:543: UserWarning: The parameter 's


Features combinées (33):
['user.followers_count', 'user.friends_count', 'user.listed_count', 'user.favourites_count', 'user.statuses_count', 'retweet_count', 'favorite_count', 'quote_count', 'reply_count', 'ratio_followers_friends', 'ratio_listed_followers', 'tweets_per_day', 'account_age_days', 'reciprocity_score', 'ratio_mention_status', 'total_tweet_engagement', 'is_verified', 'is_default_profile', 'is_default_image', 'is_protected', 'has_url', 'tweet_hour', 'tweet_is_weekend', 'text_length', 'bio_length', 'num_urls', 'num_hashtags', 'num_mentions', 'has_media', 'meta_bio_proba', 'tweet_polarity', 'tweet_subjectivity', 'meta_tweet_proba']


On ajoute les embedding par Word2vec

In [ ]:
def simple_tokenize(text):
    text = text.lower()
    text = re.sub(r'[.,;:`"\'!?()]', ' ', text)
    return [word for word in text.split() if word]

def extract_full_text(row):
    """Extrait le texte le plus complet disponible du tweet."""
    if "extended_tweet.full_text" in row and pd.notna(row["extended_tweet.full_text"]):
        return row["extended_tweet.full_text"]
    if "text" in row and pd.notna(row["text"]):
        return row["text"]
    return ""

try:
    train_data["full_text"] = train_data.apply(extract_full_text, axis=1)
    kaggle_data["full_text"] = kaggle_data.apply(extract_full_text, axis=1)
    X_full = train_data["full_text"].values
    y_full = train_data["label"].values.astype(int) 
    X_kaggle_full = kaggle_data["full_text"].values
    
    print("Taw data (X_full, y_full) ready.")

except FileNotFoundError as e:
    print(f"ERREUR: Now file found. Check the path : {e}")
    exit()

✓ Données brutes (X_full, y_full) chargées et prêtes.


In [ ]:
EMBEDDING_DIM = 250 
WINDOW_SIZE = 50
MIN_COUNT = 1


X_full_tokenized = [simple_tokenize(text) for text in X_full]
X_kaggle_tokenized = [simple_tokenize(text) for text in X_kaggle_full]

def document_vectorizer(tokens, model, dim):
    vector = np.zeros(dim)
    count = 0
    for word in tokens:
        if word in model.wv:
            vector += model.wv[word]
            count += 1
    if count != 0:
        vector /= count
        
    return vector

w2v_model = Word2Vec(
    sentences=X_full_tokenized, 
    vector_size=EMBEDDING_DIM, 
    window=WINDOW_SIZE, 
    min_count=MIN_COUNT, 
    sg=1)

X_train_vectors = np.array([document_vectorizer(tokens, w2v_model, EMBEDDING_DIM) for tokens in X_full_tokenized])
X_kaggle_vectors = np.array([document_vectorizer(tokens, w2v_model, EMBEDDING_DIM) for tokens in X_kaggle_tokenized])

embed_cols = [f'w2v_e_{i}' for i in range(EMBEDDING_DIM)]

X_train_nlp = pd.DataFrame(X_train_vectors, columns=embed_cols)
X_kaggle_nlp = pd.DataFrame(X_kaggle_vectors, columns=embed_cols)
X_train_advanced = pd.concat([X_train_advanced, X_train_nlp], axis=1)
X_kaggle_advanced = pd.concat([X_kaggle_advanced, X_kaggle_nlp], axis=1)

print(f"Total Features train (X_train_advanced) : {len(X_train_advanced.columns)}")
print(f"Total Features of test (X_kaggle_advanced) : {len(X_kaggle_advanced.columns)}")

Total Features d'entraînement (X_train_advanced) : 283
Total Features de test (X_kaggle_advanced) : 283


Xgboost final

In [12]:
optimal_params = {
    'colsample_bytree': 0.7069187275124247,
    'reg_alpha': 0.5247746602583891, 
    'reg_lambda': 1.9993048585762774, 
    'learning_rate': 0.01653319284990616,
    'min_child_weight': 79, 
    'n_estimators': 489,
    'max_depth': 91, 
    'subsample': 0.935552788417904,
}

# Best parameters
xgb_model = xgb.XGBClassifier(
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    tree_method='hist',
    colsample_bytree=optimal_params['colsample_bytree'],
    reg_alpha=optimal_params['reg_alpha'],
    reg_lambda=optimal_params['reg_lambda'],
    learning_rate=optimal_params['learning_rate'],
    min_child_weight=optimal_params['min_child_weight'],
    n_estimators=optimal_params['n_estimators'],
    max_depth=optimal_params['max_depth'],
    subsample=optimal_params['subsample'],
)

xgb_model.fit(X_train_advanced, y_train_clean)
y_pred_kaggle = xgb_model.predict(X_kaggle_advanced)

# =======================================================
# GÉNÉRATION submission .csv
# =======================================================
output = pd.concat([X_kaggle['challenge_id'], pd.DataFrame(y_pred_kaggle)], axis=1, ignore_index=True)
output.columns = ['ID', "Prediction"]
output.to_csv('submission_xgboost_without_vote.csv', index=False)

print("\n Fichier 'submission_xgboost_without_vote.csv' generated !")
print(output.head())

c:\Users\hp1ma\anaconda3\envs\inf554\Lib\site-packages\xgboost\training.py:199: UserWarning: [15:40:18] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



 Fichier 'submission_xgboost_without_vote.csv' generated !
   ID  Prediction
0   0           1
1   2           1
2   4           0
3   8           1
4   9           0


With the previous fil we get on kaggle 0.840 and by adding the following majority vote code it reachs to 0.842.

In [13]:
USER_ID_COLUMN = 'user.profile_banner_url'

df_kaggle_preds = pd.DataFrame({
    'challenge_id': X_kaggle['challenge_id'],
    'user_id_key': X_kaggle[USER_ID_COLUMN],
    'y_pred_tweet': y_pred_kaggle           # Prédiction individuelle (le secours)
})

user_pred_mean = df_kaggle_preds.groupby('user_id_key')['y_pred_tweet'].mean()
user_majority_vote = np.where(user_pred_mean >= 0.5, 1, 0)

df_majority_vote = pd.DataFrame({
    'user_id_key': user_pred_mean.index,
    'y_pred_user_majority': user_majority_vote
})

df_final_preds = pd.merge(
    df_kaggle_preds,
    df_majority_vote,
    on='user_id_key',
    how='left'
)

df_final_preds['y_pred_final'] = df_final_preds['y_pred_user_majority'].fillna(
    df_final_preds['y_pred_tweet']
)

y_final_submission = df_final_preds['y_pred_final']
output = pd.DataFrame({
    'ID': df_final_preds['challenge_id'],
    "Prediction": y_final_submission
})

output['Prediction'] = output['Prediction'].astype(int)
output.to_csv('submission_xgboost_with_majority_vote_final.csv', index=False)

print("\n Fichier 'submission_xgboost_with_majority_vote_final.csv' generated !")
print(output.head())


 Fichier 'submission_xgboost_with_majority_vote_final.csv' generated !
   ID  Prediction
0   0           1
1   2           1
2   4           0
3   8           1
4   9           0


## Examples of other text embeddings explored


Using LoRA finetuned weights on the classifying task to compute MiniLM embeddings


In [ ]:
BASE_MODEL = "nreimers/MiniLM-L6-H384-uncased"
ADAPTER_PATH = "minilm_lora_adapter"

print("Loading tokenizer and classification base model...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    torch_dtype=torch.float16
)

print("Loading LoRA adapter...")

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.to("cuda")
model.eval()

print("Model loaded on", next(model.parameters()).device)

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.hidden_states[-1]  
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).to(token_embeddings.dtype)
    return (token_embeddings * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

model.config.output_hidden_states = True

def embed_text(text_series, batch_size=32):
    embeddings = np.zeros((len(text_series), 384), dtype=np.float16)

    for i in range(0, len(text_series), batch_size):
        batch = text_series.iloc[i:i+batch_size].tolist()

        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=64,
            return_tensors="pt"
        ).to("cuda")

        with torch.no_grad():
            out = model(
                input_ids=enc["input_ids"],
                attention_mask=enc["attention_mask"],
                output_hidden_states=True
            )

        pooled = mean_pooling(out, enc["attention_mask"])

        embeddings[i:i+batch_size] = pooled.float().cpu().numpy()
        torch.cuda.empty_cache()

    return embeddings

X_train_text = get_text(X_train)
X_kaggle_text = get_text(X_kaggle)

print("Text extracted")

print("Embedding train...")
X_train_vec = embed_text(X_train_text)

print("Embedding Kaggle...")
X_kaggle_vec = embed_text(X_kaggle_text)

embed_cols = [f"e{i}" for i in range(X_train_vec.shape[1])]

X_train_nlp = pd.DataFrame(X_train_vec, columns=embed_cols)
X_kaggle_nlp = pd.DataFrame(X_kaggle_vec, columns=embed_cols)

X_train_final = pd.concat(
    [X_train_advanced.reset_index(drop=True), X_train_nlp], axis=1
)
X_kaggle_final = pd.concat(
    [X_kaggle_advanced.reset_index(drop=True), X_kaggle_nlp], axis=1
)
print("Embeddings merged")

xgb_model = xgb.XGBClassifier(
    n_estimators=250,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.8,
    tree_method="hist",
    eval_metric="mlogloss",
    random_state=42
)

print("Training XGBoost...")
xgb_model.fit(X_train_final, y_train_clean)

preds = xgb_model.predict(X_kaggle_final)

submission = pd.DataFrame({
    "ID": kaggle_data["challenge_id"],
    "Prediction": preds
})

submission.to_csv("submission_lightweight.csv", index=False)

submission.head()

#from google.colab import files
#files.download('submission_lightweight.csv')

Trying the stronger embedding model (BGE-small-en-v1.5)

In [ ]:
MODEL_NAME = "BAAI/bge-small-en-v1.5"

print("Loading BGE-small-en-v1.5...")
model = SentenceTransformer(MODEL_NAME, device="cuda")
embed_dim = model.get_sentence_embedding_dimension()
print("Model loaded on GPU — embedding dim:", embed_dim)


def get_text(df):
    if "extended_tweet.full_text" in df.columns:
        return df["extended_tweet.full_text"].fillna("").astype(str)
    if "text" in df.columns:
        return df["text"].fillna("").astype(str)
    return pd.Series([""] * len(df))


def embed_text(text_series, batch_size=64):
    texts = text_series.tolist()

    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        device="cuda"
    )

    return embeddings.astype(np.float16)


print("Extracting text...")
X_train_text = get_text(X_train)
X_kaggle_text = get_text(X_kaggle)

print("Embedding train...")
X_train_vec = embed_text(X_train_text)

print("Embedding Kaggle...")
X_kaggle_vec = embed_text(X_kaggle_text)


embed_cols = [f"e{i}" for i in range(X_train_vec.shape[1])]

X_train_nlp = pd.DataFrame(X_train_vec, columns=embed_cols)
X_kaggle_nlp = pd.DataFrame(X_kaggle_vec, columns=embed_cols)

X_train_final = pd.concat([X_train_advanced.reset_index(drop=True),
                           X_train_nlp], axis=1)

X_kaggle_final = pd.concat([X_kaggle_advanced.reset_index(drop=True),
                            X_kaggle_nlp], axis=1)

print("Embeddings merged with advanced features")

xgb_model = xgb.XGBClassifier(
    n_estimators=250,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.8,
    tree_method="hist",
    eval_metric="mlogloss",
    random_state=42
)

print("Training XGBoost...")
xgb_model.fit(X_train_final, y_train_clean)

preds = xgb_model.predict(X_kaggle_final)

submission = pd.DataFrame({
    "ID": kaggle_data["challenge_id"],
    "Prediction": preds
})

submission.to_csv("submission_bge_small.csv", index=False)

print("Saved submission_bge_small.csv")
submission.head()

#from google.colab import files
#files.download('submission_bge_small.csv')

## Obtaining feature importance at different stages of our dataset enriching 

The Transformer text embedding used here is camembert, since the tweets were written in French, we thought it was a suitable choice.

In [ ]:
def get_text(df):
    if "extended_tweet.full_text" in df.columns:
        return df["extended_tweet.full_text"].fillna("").astype(str)
    if "text" in df.columns:
        return df["text"].fillna("").astype(str)
    return pd.Series([""] * len(df), index=df.index)


train_data = load_jsonl_skip_bad("train.jsonl")

kaggle_data = pd.read_json('kaggle_test.jsonl', lines=True)
kaggle_data = json_normalize(kaggle_data.to_dict(orient='records'))

X_train = train_data.drop("label", axis=1)
y_train = train_data["label"].astype(int)
X_kaggle = kaggle_data

y_train_clean = y_train.astype(int)

X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train, y_train_clean, test_size=0.2, random_state=42, stratify=y_train_clean
)



###################################
print("STAGE 1: RAW FEATURES ONLY")
###################################

raw_cols = X_train_split.select_dtypes(include=[np.number]).columns

Xtr_raw = X_train_split[raw_cols]
Xval_raw = X_val_split[raw_cols]

Xtr_raw = Xtr_raw.drop(columns=['id', 'challenge_id'], errors='ignore')
Xval_raw = Xval_raw.drop(columns=['id', 'challenge_id'], errors='ignore')

xgb_raw = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1, tree_method="hist",
    eval_metric="logloss", random_state=42
)

xgb_raw.fit(Xtr_raw, y_train_split, eval_set=[(Xval_raw, y_val_split)], verbose=False)

raw_preds = xgb_raw.predict(Xval_raw)
raw_proba = xgb_raw.predict_proba(Xval_raw)
raw_acc = accuracy_score(y_val_split, raw_preds)
raw_loss = log_loss(y_val_split, raw_proba)

print(f" RAW Accuracy: {raw_acc:.4f}")
print(f" RAW LogLoss:  {raw_loss:.4f}")

# EXPORT STAGE 1 FEATURE IMPORTANCE
importance_raw = pd.DataFrame({
    'Feature': Xtr_raw.columns,
    'Importance_Gain': xgb_raw.feature_importances_
}).sort_values(by='Importance_Gain', ascending=False)
importance_raw.to_csv("feature_importance_stage1_raw.csv")
# files.download("feature_importance_stage1_raw.csv") # Uncomment to download
print("-> Exported feature_importance_stage1_raw.csv")





##############################################
print("STAGE 2: RAW + ENGINEERED")
##############################################

# Generate Engineered Features
X_train_adv = create_advanced_features(X_train)
X_kaggle_adv = create_advanced_features(X_kaggle)
X_train_nlp, X_kaggle_nlp = create_nlp_features(X_train, X_kaggle, y_train_clean)

# Combine Raw + Engineered Features
X_train_full_combined = pd.concat([X_train_adv.reset_index(drop=True), X_train_nlp.reset_index(drop=True)], axis=1)

Xtr_full = X_train_full_combined.iloc[X_train_split.index]
Xval_full = X_train_full_combined.iloc[X_val_split.index]

Xtr_full = Xtr_full.drop(columns=['id', 'challenge_id'], errors='ignore')
Xval_full = Xval_full.drop(columns=['id', 'challenge_id'], errors='ignore')

xgb_features = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1, tree_method="hist",
    eval_metric="logloss", random_state=42
)

xgb_features.fit(Xtr_full, y_train_split, eval_set=[(Xval_full, y_val_split)], verbose=False)

feat_preds = xgb_features.predict(Xval_full)
feat_proba = xgb_features.predict_proba(Xval_full)
feat_acc = accuracy_score(y_val_split, feat_preds)
feat_loss = log_loss(y_val_split, feat_proba)

print(f" FEATURES Accuracy: {feat_acc:.4f}")
print(f" FEATURES LogLoss:  {feat_loss:.4f}")

# EXPORT STAGE 2 FEATURE IMPORTANCE
importance_feat = pd.DataFrame({
    'Feature': Xtr_full.columns,
    'Importance_Gain': xgb_features.feature_importances_
}).sort_values(by='Importance_Gain', ascending=False)
importance_feat.to_csv("feature_importance_stage2_raw_engineered.csv")
# files.download("feature_importance_stage2_raw_engineered.csv") # Uncomment to download
print("-> Exported feature_importance_stage2_raw_engineered.csv")



###############################################################
print("STAGE 3: RAW + ENGINEERED + IMPROVED FRENCH EMBEDDINGS")
###############################################################

# 1. Define and Load the Embedding Model
EMB_MODEL = "dangvantuan/sentence-camembert-large"
print(f"Using embedding model: {EMB_MODEL}")
device = "cuda" if torch.cuda.is_available() else "cpu"
embedder = SentenceTransformer(EMB_MODEL, device=device)

# 2. Generate Embeddings for Training Data
X_train_text = get_text(X_train)

print(" Computing embeddings for Train...")
X_embed_train = embedder.encode(
    X_train_text.tolist(),
    batch_size=32,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype(np.float16)

# Create DataFrame for embeddings
embed_cols = [f"e{i}" for i in range(X_embed_train.shape[1])]
X_embed_df = pd.DataFrame(X_embed_train, columns=embed_cols)

# 3. Combine Advanced Features + NLP + Embeddings
X_train_all = pd.concat([
    X_train_full_combined.reset_index(drop=True),
    X_embed_df.reset_index(drop=True)
], axis=1)

# 4. Apply the Split (Train vs Val)
Xtr_all = X_train_all.iloc[X_train_split.index]
Xval_all = X_train_all.iloc[X_val_split.index]

# Drop IDs
Xtr_all = Xtr_all.drop(columns=['id', 'challenge_id'], errors='ignore')
Xval_all = Xval_all.drop(columns=['id', 'challenge_id'], errors='ignore')

# 5. Optimized parameters from clean0.842.ipynb
optimal_params = {
    'n_estimators': 489,
    'max_depth': 91,
    'learning_rate': 0.01653319284990616,
    'subsample': 0.935552788417904,
    'colsample_bytree': 0.7069187275124247,
    'min_child_weight': 79,
    'reg_alpha': 0.5247746602583891,
    'reg_lambda': 1.9993048585762774,
}

xgb_all = xgb.XGBClassifier(
    **optimal_params,
    tree_method="hist",
    eval_metric="logloss",
    random_state=42,
    use_label_encoder=False
)

print("Training Stage 3 XGBoost with optimized parameters...")
xgb_all.fit(Xtr_all, y_train_split, eval_set=[(Xval_all, y_val_split)], verbose=False)

all_preds = xgb_all.predict(Xval_all)
all_proba = xgb_all.predict_proba(Xval_all)
all_acc = accuracy_score(y_val_split, all_preds)
all_loss = log_loss(y_val_split, all_proba)

print(f" ALL Accuracy: {all_acc:.4f}")
print(f" ALL LogLoss:  {all_loss:.4f}")

# EXPORT STAGE 3 FEATURE IMPORTANCE
importance_all = pd.DataFrame({
    'Feature': Xtr_all.columns,
    'Importance_Gain': xgb_all.feature_importances_
}).sort_values(by='Importance_Gain', ascending=False)
importance_all.to_csv("feature_importance_stage3_full_emb.csv")
files.download("feature_importance_stage3_full_emb.csv")
print("-> Exported feature_importance_stage3_full_emb.csv")

#################################
print("FINAL SCORE COMPARISON ")
#################################

labels = ["RAW", "RAW+NLP", "FULL+EMB"]
accs = [raw_acc, feat_acc, all_acc]
losses = [raw_loss, feat_loss, all_loss]

print("Feature Set  | Accuracy  | LogLoss")
print("-------------|-----------|---------")
for l,a,o in zip(labels, accs, losses):
    print(f"{l:12} | {a:.4f}    | {o:.4f}")




####################################
print("FINAL STEP: FULL RETRAINING")
####################################

# 1. Prepare Features for Kaggle Set (Advanced + NLP)
# (Assuming X_kaggle_adv and X_kaggle_nlp were created in Stage 2)
X_kaggle_full = pd.concat([X_kaggle_adv.reset_index(drop=True), X_kaggle_nlp.reset_index(drop=True)], axis=1)

# 2. Generate Embeddings for Kaggle Set
X_kaggle_text = get_text(X_kaggle)

print(" Computing embeddings for Kaggle...")
X_embed_kaggle = embedder.encode(
    X_kaggle_text.tolist(),
    batch_size=32,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype(np.float16)

# Create DataFrame for Kaggle embeddings (using same columns as train)
X_kaggle_embed_df = pd.DataFrame(X_embed_kaggle, columns=embed_cols)

# 3. Concatenate Everything for Full Training (Train + Embeddings, Kaggle + Embeddings)
# Re-creating X_final_train from full combined data + embeddings
X_final_train = pd.concat([X_train_full_combined.reset_index(drop=True), X_embed_df.reset_index(drop=True)], axis=1)
X_final_kaggle = pd.concat([X_kaggle_full.reset_index(drop=True), X_kaggle_embed_df.reset_index(drop=True)], axis=1)

X_final_train = X_final_train.drop(columns=['id', 'challenge_id'], errors='ignore')
X_final_kaggle = X_final_kaggle.drop(columns=['id', 'challenge_id'], errors='ignore')

# 4. Train Final Model on ALL Training Data
xgb_final = xgb.XGBClassifier(
    **optimal_params,
    tree_method="hist",
    eval_metric="logloss",
    random_state=42,
    use_label_encoder=False
)

print("Training FINAL XGBoost on FULL data...")
# Note: No eval_set provided because we are using all labeled data for training
xgb_final.fit(X_final_train, y_train_clean, verbose=False)

# 5. Predict on Kaggle
final_preds = xgb_final.predict(X_final_kaggle)

submission = pd.DataFrame({
    "ID": kaggle_data["challenge_id"],
    "Prediction": final_preds
})

submission.to_csv("kaggle_submission.csv", index=False)
print("Saved kaggle_submission.csv")
#files.download('kaggle_submission.csv')



###############################
print("probability prediction")
###############################

kaggle_probas = xgb_final.predict_proba(X_kaggle_all)

# Submission file
submission = pd.DataFrame({
    "ID": kaggle_data["challenge_id"],
    "Prediction": kaggle_preds
})

submission.to_csv("kaggle_submission.csv", index=False)
files.download('kaggle_submission.csv')

# Probability file for ensemble voting
proba_submission = pd.DataFrame({
    "ID": kaggle_data["challenge_id"],
    "proba_observer": kaggle_probas[:, 0],
    "proba_influencer": kaggle_probas[:, 1]
})

proba_submission.to_csv("kaggle_submission_proba.csv", index=False)
files.download('kaggle_submission_proba.csv')

print("Saved kaggle_submission.csv")
print("Saved kaggle_submission_proba.csv")


#########################################
print("USER-LEVEL PROBABILITY AVERAGING")
#########################################

# Loading tweet-level probabilities 
probas = pd.read_csv("kaggle_submission_proba.csv")

# Extracting user key 
USER_ID_COLUMN = "user.profile_banner_url"

# Build tweet-level prediction table
df_preds = pd.DataFrame({
    "challenge_id": X_kaggle["challenge_id"].values,
    "user_id": X_kaggle[USER_ID_COLUMN].astype(str).values,
    "proba_influencer": probas["proba_influencer"].values
})

# Backup tweet-level label 
df_preds["tweet_pred"] = (df_preds["proba_influencer"] >= 0.5).astype(int)


user_mean_proba = (
    df_preds
    .groupby("user_id")["proba_influencer"]
    .mean()
    .reset_index()
)

user_mean_proba["user_pred"] = (user_mean_proba["proba_influencer"] >= 0.5).astype(int)

# Merge user-level decision back to each tweet
df_final = df_preds.merge(
    user_mean_proba[["user_id", "user_pred"]],
    on="user_id",
    how="left"
)

# Final fallback protection (single-tweet users)
df_final["final_prediction"] = (
    df_final["user_pred"]
    .fillna(df_final["tweet_pred"])
    .astype(int)
)

# Final Kaggle submission
submission_user_vote = pd.DataFrame({
    "ID": df_final["challenge_id"],
    "Prediction": df_final["final_prediction"]
})

submission_user_vote.to_csv("kaggle_submission_user_vote.csv", index=False)

print("Saved: kaggle_submission_user_vote.csv")
#files.download('kaggle_submission_user_vote.csv')

## Second approach : full deep learning pipeline --> see other notebook

## Combining probabilities from our XGBoost pipeline with the probabilities obtained with the full deep learning paradigm 

In [ ]:
# ====================================================================
# 1. LOAD AND PREPARE PROBABILITIES (XGBoost Model)
# ====================================================================

# Load XGBoost probabilities from the file generated in the previous step
df_proba_xgb = pd.read_csv("kaggle_submission_proba.csv")
df_proba_xgb = df_proba_xgb[['ID', 'proba_influencer']].rename(columns={
    'proba_influencer': 'P_xgb_1'
})

# ====================================================================
# 2. Deep learning model probabilities (apply softmax)
# ====================================================================

# Load deep learning model scores
df_scores_new_model = pd.read_csv("PROBAS_DL.csv")

# Define softmax function (ensuring numerical stability with max on axis=1)
def softmax(x):
    e_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e_x / np.sum(e_x, axis=1, keepdims=True)

# Using the actual column names provided by the user
score_columns_to_process = ['y_pred_proba_observer', 'y_pred_proba_influencer']

# Ensure the score columns exist before proceeding
for col in score_columns_to_process:
    if col not in df_scores_new_model.columns:
        raise KeyError(f"Column '{col}' not found in PROBAS_DL.csv. Please verify column names.")

scores = df_scores_new_model[score_columns_to_process].values

# Apply softmax to get probabilities for both classes
probabilities = softmax(scores)

# Create df_proba_new_model with 'ID' and 'P_new_1' (influencer probability)
df_proba_new_model = pd.DataFrame({
    'ID': df_scores_new_model['challenge_id'],
    'P_new_1': probabilities[:, 1] # P_new_1 will be the probability of the influencer class (index 1)
})

# ====================================================================
# 3. SOFT VOTING ENSEMBLE (TWEET-LEVEL)
# ====================================================================

# Merge the probabilities based on the unique identifier (ID)
df_ensemble = df_proba_xgb[['ID', 'P_xgb_1']].merge(
    df_proba_new_model[['ID', 'P_new_1']],
    on='ID',
    how='inner'
)

# Calculate the simple average (Soft Vote) for Class 1 (influencer)
df_ensemble['P_ensemble_1'] = (df_ensemble['P_xgb_1'] + df_ensemble['P_new_1']) / 2

# Final binary prediction based on ensemble probability
df_ensemble['Prediction_TWEET_LEVEL'] = (df_ensemble['P_ensemble_1'] >= 0.5).astype(int)

# ====================================================================
# 4. FIRST OUTPUT: TWEET-LEVEL ENSEMBLE SUBMISSION (NO AGGREGATION)
# ====================================================================

submission_tweet_level = pd.DataFrame({
    'ID': df_ensemble['ID'],
    'Prediction': df_ensemble['Prediction_TWEET_LEVEL']
})

FILE_TWEET_LEVEL = "kaggle_submission_ensemble_tweet_level.csv"
submission_tweet_level.to_csv(FILE_TWEET_LEVEL, index=False)
print(f"Saved TWEET-LEVEL Ensemble Submission: {FILE_TWEET_LEVEL}")
files.download(FILE_TWEET_LEVEL)


# ====================================================================
# 5. USER-LEVEL AGGREGATION (Final Step)
# ====================================================================

X_kaggle = pd.read_json("kaggle_test.jsonl",lines=True)

USER_ID_COLUMN = "user.profile_banner_url"

# Prepare the data for aggregation: ensemble probability + user ID
df_preds = pd.DataFrame({
    "challenge_id": X_kaggle["challenge_id"].values,
    "user_id": X_kaggle[USER_ID_COLUMN].astype(str).values,
    "proba_influencer": df_ensemble['P_ensemble_1'].values # Use the ensemble probability
})

# USER-LEVEL PROBABILITY AVERAGING
user_mean_proba = (
    df_preds
    .groupby("user_id")["proba_influencer"]
    .mean() # Average the ensemble probability across all user's tweets
    .reset_index()
)

# User-level binary prediction
user_mean_proba["user_pred"] = (user_mean_proba["proba_influencer"] >= 0.5).astype(int)

# Merge user-level decision back to each tweet
df_final = df_preds.merge(
    user_mean_proba[["user_id", "user_pred"]],
    on="user_id",
    how="left"
)

df_final["final_prediction"] = df_final["user_pred"].astype(int)

# ====================================================================
# 6. SECOND OUTPUT: USER-LEVEL AGGREGATED ENSEMBLE SUBMISSION
# ====================================================================

submission_user_vote = pd.DataFrame({
    "ID": df_final["challenge_id"],
    "Prediction": df_final["final_prediction"]
})

FILE_USER_LEVEL = "kaggle_submission_ensemble_user_vote.csv"
submission_user_vote.to_csv(FILE_USER_LEVEL, index=False)
print(f"Saved USER-LEVEL Aggregated Ensemble Submission: {FILE_USER_LEVEL}")
#files.download(FILE_USER_LEVEL)